In [1]:
# ============================================================
# M5.4 — WARNING-FREE LIMIT CONSTANT AUDIT FOR LEMMA B
# ============================================================
#
# Goal:
#   Compute the infinite-cover renormalized limiting constant
#
#       C_inf_star
#       = lim_{mu->0} [
#           Y_inf(mu^2)
#           - A log(L/4)
#           - (A/2) log beta
#         ]
#
#   without scipy improper-integral roundoff warnings.
#
# The exact reduction is:
#
#   Y_inf(mu^2)
#   = (3/4) ∫_0^∞ t exp(-mu^2 t) q(t)^4 dt
#
# where
#
#   q(t)=exp(-2t) I_0(2t).
#
# Since
#
#   q(t)^4 ~ 1/(16*pi^2*t^2),
#
# set
#
#   c = 1/(16*pi^2).
#
# Then
#
#   C_inf_star
#   =
#   (3/4) [
#       ∫_0^T0 t q(t)^4 dt
#       + ∫_T0^∞ (t q(t)^4 - c/t) dt
#       + c(-EulerGamma - log T0)
#   ]
#   + A log 4
#   - (A/2) log 48.
#
# This script computes the residual integral over finite geometric intervals,
# then estimates the remaining tail from the known asymptotic scale.
#
# This is still an audit, not a formal interval-arithmetic proof.
# It is designed to make the M5 proof target numerically stable and clean.
# ============================================================

import math
import time
import numpy as np

try:
    from scipy.special import ive
    from scipy.integrate import quad
except Exception as e:
    raise RuntimeError("SciPy is required. In Colab, run: !pip install scipy") from e

print("=" * 100)
print("M5.4 WARNING-FREE LIMIT CONSTANT AUDIT")
print("=" * 100)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------

A = 3.0 / (32.0 * math.pi**2)
c = 1.0 / (16.0 * math.pi**2)
EulerGamma = 0.577215664901532860606512090082402431

C_CRIT = 0.036241888089666857

T0 = 64.0
TMAX = 2.0**24   # 16,777,216; residual tail after this is tiny.

print(f"A                 = {A:.18f}")
print(f"c                 = {c:.18f}")
print(f"C_CRIT            = {C_CRIT:.18f}")
print(f"T0                = {T0:g}")
print(f"TMAX              = {TMAX:g}")
print()

# ------------------------------------------------------------
# Heat kernel q(t)
# ------------------------------------------------------------

def q(t):
    return float(ive(0, 2.0 * t))

def main_integrand(t):
    qt = q(t)
    return t * qt**4

def residual_integrand(t):
    qt = q(t)
    return t * qt**4 - c / t

# ------------------------------------------------------------
# Direct finite part
# ------------------------------------------------------------

start = time.time()

I0, E0 = quad(
    main_integrand,
    0.0,
    T0,
    epsabs=2e-14,
    epsrel=2e-14,
    limit=500
)

print("FINITE CORE")
print("-" * 100)
print(f"I0 = ∫_0^T0 t q(t)^4 dt = {I0:.18f}")
print(f"quad err                    = {E0:.3e}")
print()

# ------------------------------------------------------------
# Residual integral on finite geometric intervals
# ------------------------------------------------------------

print("RESIDUAL GEOMETRIC INTEGRAL")
print("-" * 100)

residual_sum = 0.0
residual_err = 0.0

a = T0
interval_rows = []

while a < TMAX:
    b = min(2.0 * a, TMAX)

    val, err = quad(
        residual_integrand,
        a,
        b,
        epsabs=1e-16,
        epsrel=2e-13,
        limit=300
    )

    residual_sum += val
    residual_err += abs(err)

    interval_rows.append((a, b, val, err))

    # Print a compressed trace.
    if a in [T0, 128.0, 256.0, 512.0, 1024.0, 2048.0, 4096.0, 8192.0, 16384.0]:
        print(f"[{a:10.1f}, {b:10.1f}]  val={val:.18e}  err={err:.3e}")

    a = b

print(f"... {len(interval_rows)} intervals total")
print(f"residual finite sum          = {residual_sum:.18f}")
print(f"residual accumulated err     = {residual_err:.3e}")
print()

# ------------------------------------------------------------
# Conservative asymptotic tail estimate
# ------------------------------------------------------------
#
# Asymptotic:
#   q(t) = (4*pi*t)^(-1/2) [1 + 1/(16t) + O(t^-2)].
#
# Therefore:
#   t q(t)^4 - c/t = c/(4t^2) + O(t^-3).
#
# Tail after TMAX is approximately c/(4*TMAX).
# We use a deliberately inflated bound: c/TMAX.
# ------------------------------------------------------------

tail_est = c / (4.0 * TMAX)
tail_bound = c / TMAX

print("ASYMPTOTIC RESIDUAL TAIL")
print("-" * 100)
print(f"tail estimate c/(4*TMAX)     = {tail_est:.18e}")
print(f"conservative bound c/TMAX    = {tail_bound:.18e}")
print()

# ------------------------------------------------------------
# Limit constant
# ------------------------------------------------------------

renorm_piece = c * (-EulerGamma - math.log(T0))

C_star_center = (
    0.75 * (I0 + residual_sum + renorm_piece)
    + A * math.log(4.0)
    - 0.5 * A * math.log(48.0)
)

C_star_upper = C_star_center + 0.75 * (residual_err + tail_bound + abs(E0))
C_star_lower = C_star_center - 0.75 * (residual_err + tail_bound + abs(E0))

elapsed = time.time() - start

print("=" * 100)
print("LIMIT CONSTANT RESULT")
print("=" * 100)
print(f"renorm piece c(-gamma-logT0) = {renorm_piece:.18f}")
print(f"C_inf_star center           = {C_star_center:.18f}")
print(f"C_inf_star lower            = {C_star_lower:.18f}")
print(f"C_inf_star upper            = {C_star_upper:.18f}")
print(f"C_CRIT                      = {C_CRIT:.18f}")
print(f"margin using upper          = {C_CRIT - C_star_upper:.18f}")
print(f"safe 0.013 pass?            = {C_star_upper < 0.013}")
print(f"critical pass?              = {C_star_upper < C_CRIT}")
print(f"elapsed seconds             = {elapsed:.2f}")
print()

# ------------------------------------------------------------
# Consequence for T_C envelope if Csafe = 0.013
# ------------------------------------------------------------

BETA0 = 5.6
GAMMA = 11.0 / (8.0 * math.pi**2)

def beta_x(x):
    return BETA0 + GAMMA * x

def TC_envelope(x, Csafe):
    b = beta_x(x)
    return 36.0 * (A*x + 0.5*A*math.log(b) + Csafe) / (b*b)

def maximize_envelope(Csafe, xmax=200.0):
    lo, hi = 0.0, xmax

    for _ in range(240):
        m1 = lo + (hi-lo)/3.0
        m2 = hi - (hi-lo)/3.0

        if TC_envelope(m1, Csafe) < TC_envelope(m2, Csafe):
            lo = m1
        else:
            hi = m2

    xs = 0.5*(lo+hi)
    return xs, beta_x(xs), TC_envelope(xs, Csafe)

print("=" * 100)
print("CEILING CONSEQUENCE WITH CSAFE = 0.013")
print("=" * 100)

for Csafe in [C_star_upper, 0.013, 0.015, 0.020, C_CRIT]:
    xs, bs, tb = maximize_envelope(Csafe)
    print(
        f"Csafe={Csafe:.15f}  "
        f"x*={xs:.9f}  "
        f"beta*={bs:.9f}  "
        f"Tbar={tb:.15f}  "
        f"floor(1/Tbar)={int(1.0 // tb):3d}"
    )

print()
print("M5.4 INTERPRETATION")
print("-" * 100)
print("If Lemma A proves finite-torus excess heat-kernel domination by the infinite cover,")
print("and Lemma B proves C_inf_star <= 0.013, then the deterministic M5 ceiling closes:")
print()
print("    T_C < 1/8")
print("    N*_C >= 8")
print("    T_full < 9/64")
print("    N* >= 7")
print()
print("Remaining formal proof tasks:")
print("  1. Prove cover domination: Phi_L(t)^4 - L^-4 <= q(t)^4.")
print("  2. Replace the numerical residual integral above by interval arithmetic or")
print("     explicit Bessel asymptotic inequalities.")
print("=" * 100)

M5.4 WARNING-FREE LIMIT CONSTANT AUDIT
A                 = 0.009498860966469166
c                 = 0.006332573977646111
C_CRIT            = 0.036241888089666857
T0                = 64
TMAX              = 1.67772e+07

FINITE CORE
----------------------------------------------------------------------------------------------------
I0 = ∫_0^T0 t q(t)^4 dt = 0.053980067308111709
quad err                    = 6.207e-16

RESIDUAL GEOMETRIC INTEGRAL
----------------------------------------------------------------------------------------------------
[      64.0,      128.0]  val=1.242302262862761735e-05  err=1.379e-19
[     128.0,      256.0]  val=6.197787437827373564e-06  err=6.881e-20
[     256.0,      512.0]  val=3.095479798456940430e-06  err=3.437e-20
[     512.0,     1024.0]  val=1.546888532736925730e-06  err=1.717e-20
[    1024.0,     2048.0]  val=7.732316878708559883e-07  err=8.585e-21
[    2048.0,     4096.0]  val=3.865627321346983019e-07  err=4.292e-21
[    4096.0,     8192.0]  val=1.

In [2]:
# ============================================================
# M5.5 — THETA-LIMIT AUDIT FOR COVER DOMINATION
# ============================================================
#
# Lemma A candidate:
#
#   Phi_L(t)^4 - L^-4 <= q(t)^4
#
# Under the diffusive scaling t = tau L^2, L -> infinity:
#
#   Phi_L(t) ~ L^-1 Theta(tau)
#   q(t)     ~ L^-1 (4*pi*tau)^(-1/2)
#
# where
#
#   Theta(tau)
#   = sum_{n in Z} exp(-4*pi^2*n^2*tau)
#   = (4*pi*tau)^(-1/2) sum_{m in Z} exp(-m^2/(4*tau)).
#
# The limiting inequality is:
#
#   Theta(tau)^4 - 1 <= (4*pi*tau)^(-2)
#
# equivalently
#
#   R(tau) := 16*pi^2*tau^2 * (Theta(tau)^4 - 1) <= 1.
#
# This block audits R(tau) over many scales and locates its maximum.
# It also prints a proof-oriented interval split.
# ============================================================

import math
import numpy as np

try:
    from scipy.optimize import minimize_scalar
except Exception as e:
    raise RuntimeError("SciPy is required. In Colab, run: !pip install scipy") from e

print("=" * 100)
print("M5.5 THETA-LIMIT AUDIT FOR COVER DOMINATION")
print("=" * 100)

PI = math.pi

# ------------------------------------------------------------
# Theta function, stable in both regimes
# ------------------------------------------------------------

def theta_spectral(tau, tol=1e-18):
    """
    Theta(tau) = sum_{n in Z} exp(-4*pi^2*n^2*tau).
    Good for moderate/large tau.
    """
    s = 1.0
    n = 1
    while True:
        term = 2.0 * math.exp(-4.0 * PI * PI * n * n * tau)
        s_new = s + term
        if term <= tol * max(1.0, abs(s_new)):
            return s_new
        s = s_new
        n += 1
        if n > 10_000_000:
            raise RuntimeError("theta_spectral runaway")

def theta_spatial(tau, tol=1e-18):
    """
    Theta(tau) = (4*pi*tau)^(-1/2) sum_{m in Z} exp(-m^2/(4*tau)).
    Good for small tau.
    """
    pref = 1.0 / math.sqrt(4.0 * PI * tau)
    s = 1.0
    m = 1
    while True:
        term = 2.0 * math.exp(-(m * m) / (4.0 * tau))
        s_new = s + term
        if pref * term <= tol * max(1.0, pref * abs(s_new)):
            return pref * s_new
        s = s_new
        m += 1
        if m > 10_000_000:
            raise RuntimeError("theta_spatial runaway")

def theta(tau):
    # Poisson switch near tau ~ 1/(4*pi) is numerically stable.
    if tau < 0.08:
        return theta_spatial(tau)
    return theta_spectral(tau)

def R(tau):
    th = theta(tau)
    return 16.0 * PI * PI * tau * tau * (th**4 - 1.0)

def gap(tau):
    """
    Positive gap means failure.
    """
    th = theta(tau)
    return th**4 - 1.0 - 1.0 / ((4.0 * PI * tau)**2)

# ------------------------------------------------------------
# Multiscale grid audit
# ------------------------------------------------------------

taus = np.logspace(-6, 6, 4000)

max_R = -1.0
max_tau = None
max_gap = -1e300
max_gap_tau = None

for tau in taus:
    tau = float(tau)
    rv = R(tau)
    gv = gap(tau)

    if rv > max_R:
        max_R = rv
        max_tau = tau

    if gv > max_gap:
        max_gap = gv
        max_gap_tau = tau

print("GRID AUDIT")
print("-" * 100)
print(f"max R on grid      = {max_R:.18f}")
print(f"at tau             = {max_tau:.18e}")
print(f"max gap on grid    = {max_gap:.18e}")
print(f"at tau             = {max_gap_tau:.18e}")
print(f"grid pass?         = {max_gap <= 1e-14}")
print()

# ------------------------------------------------------------
# Optimize on log intervals
# ------------------------------------------------------------

print("LOG-INTERVAL OPTIMIZATION")
print("-" * 100)

# Work in y = log(tau), maximize R(exp(y)).
def neg_R_y(y):
    return -R(math.exp(y))

intervals = [
    (-30, -20),
    (-20, -15),
    (-15, -10),
    (-10, -7),
    (-7, -5),
    (-5, -3),
    (-3, -1),
    (-1, 0),
    (0, 1),
    (1, 3),
    (3, 5),
    (5, 10),
    (10, 20),
    (20, 30),
]

opt_rows = []
for a, b in intervals:
    res = minimize_scalar(neg_R_y, bounds=(a, b), method="bounded", options={"xatol": 1e-13})
    ystar = res.x
    tau_star = math.exp(ystar)
    rstar = R(tau_star)
    gstar = gap(tau_star)

    opt_rows.append((a, b, tau_star, rstar, gstar))

    print(
        f"y∈[{a:6.1f},{b:6.1f}]  "
        f"tau*={tau_star:.6e}  "
        f"Rmax={rstar:.18f}  "
        f"gap={gstar:.3e}"
    )

best = max(opt_rows, key=lambda z: z[3])

print()
print("OPTIMIZATION SUMMARY")
print("-" * 100)
print(f"best interval       = [{best[0]}, {best[1]}]")
print(f"best tau            = {best[2]:.18e}")
print(f"best R              = {best[3]:.18f}")
print(f"best gap            = {best[4]:.18e}")
print(f"theta-limit pass?   = {best[3] <= 1.0 + 1e-13}")
print()

# ------------------------------------------------------------
# Proof-oriented regime split
# ------------------------------------------------------------

print("=" * 100)
print("PROOF-ORIENTED REGIME SPLIT")
print("=" * 100)

# Small tau:
#   Theta = a(1+epsilon), a=(4*pi*tau)^-1/2
#   Need a^4[(1+epsilon)^4 - 1] <= 1
#   equivalently epsilon small enough.
#
# Large tau:
#   Theta = 1 + eta, eta <= 2 exp(-4*pi^2 tau)/(1-exp(-12*pi^2 tau))
#   Need (1+eta)^4 - 1 <= (4*pi*tau)^-2.
#
# This prints conservative diagnostics for candidate split points.

def eps_small_bound(tau):
    """
    epsilon = spatial image sum excluding m=0:
        epsilon <= 2 exp(-1/(4tau)) / (1 - exp(-3/(4tau)))
    because m^2 >= 1 + 3(m-1) for m>=1.
    """
    r = math.exp(-3.0 / (4.0 * tau))
    return 2.0 * math.exp(-1.0 / (4.0 * tau)) / max(1.0 - r, 1e-300)

def small_tau_sufficient_margin(tau):
    """
    For Theta=a(1+eps), target is:
        a^4(1+eps)^4 - 1 <= a^4
    i.e.
        (1+eps)^4 - 1 <= 1/a^4 = 16*pi^2*tau^2.
    Return RHS-LHS.
    """
    eps = eps_small_bound(tau)
    lhs = (1.0 + eps)**4 - 1.0
    rhs = 16.0 * PI * PI * tau * tau
    return rhs - lhs

def eta_large_bound(tau):
    """
    eta = spectral tail excluding n=0:
        eta <= 2 exp(-4*pi^2 tau)/(1-exp(-12*pi^2 tau))
    because n^2 >= 1 + 3(n-1) for n>=1.
    """
    r = math.exp(-12.0 * PI * PI * tau)
    return 2.0 * math.exp(-4.0 * PI * PI * tau) / max(1.0 - r, 1e-300)

def large_tau_sufficient_margin(tau):
    """
    Need:
        (1+eta)^4 - 1 <= (4*pi*tau)^-2.
    Return RHS-LHS.
    """
    eta = eta_large_bound(tau)
    lhs = (1.0 + eta)**4 - 1.0
    rhs = 1.0 / ((4.0 * PI * tau)**2)
    return rhs - lhs

small_candidates = [0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 0.05]
large_candidates = [0.08, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30, 0.40]

print("Small-tau sufficient bound: margin >= 0 proves tau <= split")
for tau in small_candidates:
    print(
        f"tau={tau:8.5f}  "
        f"margin={small_tau_sufficient_margin(tau): .6e}  "
        f"eps_bound={eps_small_bound(tau):.3e}"
    )

print()
print("Large-tau sufficient bound: margin >= 0 proves tau >= split")
for tau in large_candidates:
    print(
        f"tau={tau:8.5f}  "
        f"margin={large_tau_sufficient_margin(tau): .6e}  "
        f"eta_bound={eta_large_bound(tau):.3e}"
    )

print()
print("Middle interval target")
print("-" * 100)
print("If the sufficient bounds prove small tau up to tau_s and large tau from tau_l onward,")
print("then only the compact interval [tau_s, tau_l] remains.")
print("That compact interval can be closed by interval arithmetic on theta(tau),")
print("or by monotonicity/convexity of R(tau).")
print()
print("Practical target from the diagnostics:")
print("  small tau: prove up to roughly tau = 0.03 or 0.04")
print("  large tau: prove from roughly tau = 0.10 or 0.12")
print("  middle: interval-check R(tau) <= 1 on [0.03, 0.12]")
print("=" * 100)

M5.5 THETA-LIMIT AUDIT FOR COVER DOMINATION
GRID AUDIT
----------------------------------------------------------------------------------------------------
max R on grid      = 0.999999999842086540
at tau             = 9.999999999999999547e-07
max gap on grid    = -6.332573977646111185e-15
at tau             = 1.000000000000000000e+06
grid pass?         = True

LOG-INTERVAL OPTIMIZATION
----------------------------------------------------------------------------------------------------
y∈[ -30.0, -20.0]  tau*=4.476667e-11  Rmax=1.000000000000000444  gap=1.024e+03
y∈[ -20.0, -15.0]  tau*=2.410325e-09  Rmax=0.999999999999999556  gap=-2.500e-01
y∈[ -15.0, -10.0]  tau*=3.059049e-07  Rmax=0.999999999985223265  gap=-1.000e+00
y∈[ -10.0,  -7.0]  tau*=4.539994e-05  Rmax=0.999999674515527293  gap=-1.000e+00
y∈[  -7.0,  -5.0]  tau*=9.118821e-04  Rmax=0.999868690206959454  gap=-1.000e+00
y∈[  -5.0,  -3.0]  tau*=6.737948e-03  Rmax=0.992830729230518405  gap=-1.000e+00
y∈[  -3.0,  -1.0]  tau*=4.9787

In [3]:
# ============================================================
# M5.6 — THETA-LIMIT BOX CERTIFICATE
# ============================================================
#
# Goal:
#   Certify the continuum theta-limit inequality
#
#       R(tau) = 16*pi^2*tau^2*(Theta(tau)^4 - 1) <= 1
#
#   for all tau > 0.
#
# Proof split:
#
#   1. Small tau, 0 < tau <= 0.05:
#        Use spatial theta:
#        Theta = a(1+eps), a=(4*pi*tau)^(-1/2)
#        eps <= 3 exp(-1/(4tau)).
#
#        It is enough to show
#        (1+eps)^4 - 1 <= 16*pi^2*tau^2.
#
#        For eps <= 1:
#        (1+eps)^4 - 1 <= 15 eps <= 45 exp(-1/(4tau)).
#
#        Then check
#        45 exp(-1/(4tau)) <= 16*pi^2*tau^2.
#
#        The log-ratio is increasing on tau < 1/8, so it is enough
#        to check tau = 0.05.
#
#   2. Large tau, tau >= 0.4:
#        Use spectral theta:
#        Theta = 1+eta
#        eta <= 3 exp(-4*pi^2*tau).
#
#        For eta <= 1:
#        (1+eta)^4 - 1 <= 15 eta <= 45 exp(-4*pi^2*tau).
#
#        Need
#        45 exp(-4*pi^2*tau) <= (4*pi*tau)^(-2).
#
#        The log-ratio is decreasing for tau > 1/(2*pi^2),
#        so it is enough to check tau = 0.4.
#
#   3. Middle interval [0.05, 0.4]:
#        Adaptive conservative boxes:
#
#        For tau in [a,b],
#           Theta(tau) <= Theta_upper(a)
#           tau^2 <= b^2
#
#        so
#           R(tau) <= 16*pi^2*b^2*(Theta_upper(a)^4 - 1).
#
#        If this upper bound is <= 1 for every box, the interval is certified.
#
# This is still a numerical certificate, but it avoids the cancellation failure
# in direct gap(tau) and produces explicit proof intervals.
# ============================================================

import math
import time

try:
    import mpmath as mp
except Exception as e:
    raise RuntimeError("mpmath is required. In Colab, run: !pip install mpmath") from e

mp.mp.dps = 80

print("=" * 100)
print("M5.6 THETA-LIMIT BOX CERTIFICATE")
print("=" * 100)

PI = mp.pi

SMALL_SPLIT = mp.mpf("0.05")
LARGE_SPLIT = mp.mpf("0.4")

print(f"mp precision     = {mp.mp.dps} decimal digits")
print(f"small split      = {SMALL_SPLIT}")
print(f"large split      = {LARGE_SPLIT}")
print()

# ------------------------------------------------------------
# Small tau proof check
# ------------------------------------------------------------

def small_log_margin(tau):
    """
    log(rhs/lhs_bound) for:
        45 exp(-1/(4tau)) <= 16*pi^2*tau^2.
    Positive margin proves the crude sufficient inequality.
    """
    tau = mp.mpf(tau)
    return mp.log(16 * PI**2 * tau**2) - (mp.log(45) - 1/(4*tau))

def small_eps_bound(tau):
    tau = mp.mpf(tau)
    return 3 * mp.e**(-1/(4*tau))

small_margin = small_log_margin(SMALL_SPLIT)
small_eps = small_eps_bound(SMALL_SPLIT)

print("SMALL-TAU REGIME")
print("-" * 100)
print("Claim: for 0 < tau <= 0.05, R(tau) <= 1.")
print(f"eps_bound at split              = {mp.nstr(small_eps, 30)}")
print(f"log margin at tau=0.05          = {mp.nstr(small_margin, 30)}")
print("log margin positive?            =", small_margin > 0)
print("eps_bound <= 1?                 =", small_eps <= 1)
print("monotonic note                  = log ratio is increasing on tau < 1/8")
print("small-tau certified?            =", (small_margin > 0 and small_eps <= 1))
print()

# ------------------------------------------------------------
# Large tau proof check
# ------------------------------------------------------------

def large_log_margin(tau):
    """
    log(rhs/lhs_bound) for:
        45 exp(-4*pi^2*tau) <= (4*pi*tau)^(-2).
    Positive margin proves the crude sufficient inequality.
    """
    tau = mp.mpf(tau)
    return -2 * mp.log(4 * PI * tau) - (mp.log(45) - 4 * PI**2 * tau)

def large_eta_bound(tau):
    tau = mp.mpf(tau)
    return 3 * mp.e**(-4 * PI**2 * tau)

large_margin = large_log_margin(LARGE_SPLIT)
large_eta = large_eta_bound(LARGE_SPLIT)

print("LARGE-TAU REGIME")
print("-" * 100)
print("Claim: for tau >= 0.4, R(tau) <= 1.")
print(f"eta_bound at split              = {mp.nstr(large_eta, 30)}")
print(f"log margin at tau=0.4           = {mp.nstr(large_margin, 30)}")
print("log margin positive?            =", large_margin > 0)
print("eta_bound <= 1?                 =", large_eta <= 1)
print("monotonic note                  = log ratio decreases for tau > 1/(2*pi^2)")
print("large-tau certified?            =", (large_margin > 0 and large_eta <= 1))
print()

# ------------------------------------------------------------
# Theta upper bound for middle boxes
# ------------------------------------------------------------

def theta_upper_spectral(tau, term_tol=mp.mpf("1e-70")):
    """
    Rigorous-style upper bound for:
        Theta(tau) = 1 + 2 sum_{n>=1} exp(-4*pi^2*n^2*tau).

    Uses a geometric tail after the first term below term_tol.

    For n >= N+1:
       exp(-lambda n^2)
    has consecutive ratios bounded by
       exp(-lambda*(2N+3))
    starting at n=N+1.

    Tail <= next_term/(1-ratio).
    """
    tau = mp.mpf(tau)
    lam = 4 * PI**2 * tau

    s = mp.mpf(1)
    n = 1

    while True:
        term = mp.e**(-lam * n * n)
        s += 2 * term

        if term < term_tol:
            next_n = n + 1
            next_term = mp.e**(-lam * next_n * next_n)
            ratio = mp.e**(-lam * (2 * next_n + 1))
            tail = 2 * next_term / (1 - ratio)
            return s + tail

        n += 1
        if n > 100000:
            raise RuntimeError("theta_upper_spectral runaway")

def R_box_upper(a, b):
    """
    Conservative upper bound for R(tau) on [a,b]:
        R <= 16*pi^2*b^2*(Theta_upper(a)^4 - 1).
    """
    a = mp.mpf(a)
    b = mp.mpf(b)
    th_u = theta_upper_spectral(a)
    return 16 * PI**2 * b**2 * (th_u**4 - 1)

# ------------------------------------------------------------
# Adaptive middle interval certification
# ------------------------------------------------------------

print("MIDDLE-INTERVAL BOX CERTIFICATE")
print("-" * 100)

start = time.time()

stack = [(SMALL_SPLIT, LARGE_SPLIT)]
certified = []
failed = []

max_upper = mp.mpf("-inf")
max_upper_box = None

MAX_BOXES = 2_000_000
MIN_WIDTH = mp.mpf("1e-18")

while stack:
    a, b = stack.pop()

    upper = R_box_upper(a, b)

    if upper > max_upper:
        max_upper = upper
        max_upper_box = (a, b, upper)

    if upper <= 1:
        certified.append((a, b, upper))
        continue

    width = b - a

    if width <= MIN_WIDTH or len(certified) + len(stack) > MAX_BOXES:
        failed.append((a, b, upper))
        break

    mid = (a + b) / 2
    stack.append((mid, b))
    stack.append((a, mid))

elapsed = time.time() - start

middle_pass = (len(failed) == 0)

print(f"middle certified?              = {middle_pass}")
print(f"certified boxes                = {len(certified)}")
print(f"failed boxes                   = {len(failed)}")
print(f"elapsed seconds                = {elapsed:.3f}")
print(f"max interval upper R           = {mp.nstr(max_upper, 30)}")
print(f"max upper box                  = [{mp.nstr(max_upper_box[0], 20)}, {mp.nstr(max_upper_box[1], 20)}]")
print()

if failed:
    print("FAILED BOX EXAMPLE")
    print("-" * 100)
    a, b, u = failed[0]
    print(f"a={mp.nstr(a, 30)}")
    print(f"b={mp.nstr(b, 30)}")
    print(f"upper={mp.nstr(u, 30)}")
    print()

# ------------------------------------------------------------
# Independent point audit with high precision, no subtractive gap
# ------------------------------------------------------------

def theta_exact(tau):
    """
    High precision theta by spectral sum.
    """
    return theta_upper_spectral(tau, term_tol=mp.mpf("1e-75"))

def R_exact(tau):
    tau = mp.mpf(tau)
    th = theta_exact(tau)
    return 16 * PI**2 * tau**2 * (th**4 - 1)

print("HIGH-PRECISION POINT AUDIT")
print("-" * 100)

audit_points = [
    mp.mpf("1e-12"),
    mp.mpf("1e-9"),
    mp.mpf("1e-6"),
    mp.mpf("1e-4"),
    mp.mpf("0.001"),
    mp.mpf("0.005"),
    mp.mpf("0.01"),
    mp.mpf("0.02"),
    mp.mpf("0.03"),
    mp.mpf("0.04"),
    mp.mpf("0.05"),
    mp.mpf("0.075"),
    mp.mpf("0.10"),
    mp.mpf("0.12"),
    mp.mpf("0.20"),
    mp.mpf("0.40"),
    mp.mpf("1.0"),
    mp.mpf("10.0"),
]

for tau in audit_points:
    rv = R_exact(tau)
    print(
        f"tau={mp.nstr(tau, 12):>14s}  "
        f"R={mp.nstr(rv, 30):>34s}  "
        f"1-R={mp.nstr(1-rv, 20)}"
    )

print()

# ------------------------------------------------------------
# Final theorem status
# ------------------------------------------------------------

overall = (
    small_margin > 0
    and small_eps <= 1
    and large_margin > 0
    and large_eta <= 1
    and middle_pass
)

print("=" * 100)
print("M5.6 RESULT")
print("=" * 100)
print(f"small regime certified?        = {small_margin > 0 and small_eps <= 1}")
print(f"middle regime certified?       = {middle_pass}")
print(f"large regime certified?        = {large_margin > 0 and large_eta <= 1}")
print(f"theta-limit inequality boxed?  = {overall}")
print()

if overall:
    print("Theta-limit obstruction is closed at certificate level:")
    print()
    print("    16*pi^2*tau^2*(Theta(tau)^4 - 1) <= 1  for all tau > 0.")
    print()
    print("This supports the cover-domination route, but does not yet prove")
    print("the finite-L inequality Phi_L(t)^4 - L^-4 <= q(t)^4.")
else:
    print("Theta-limit certificate did not close. Inspect failed boxes or margins.")

print()
print("NEXT FORMAL PIN AFTER M5.6")
print("-" * 100)
print("Prove the finite-cycle heat-kernel excess domination:")
print()
print("    Phi_L(t)^4 - L^-4 <= q(t)^4")
print()
print("for every integer L >= 4 and t > 0.")
print()
print("Then M5 closes with Csafe=0.013 and yields:")
print("    T_C < 1/8,  N*_C >= 8,  T_full < 9/64,  N* >= 7.")
print("=" * 100)

M5.6 THETA-LIMIT BOX CERTIFICATE
mp precision     = 80 decimal digits
small split      = 0.05
large split      = 0.4

SMALL-TAU REGIME
----------------------------------------------------------------------------------------------------
Claim: for 0 < tau <= 0.05, R(tau) <= 1.
eps_bound at split              = 0.0202138409972564012899081452694
log margin at tau=0.05          = 0.263921457060279841694086229183
log margin positive?            = True
eps_bound <= 1?                 = True
monotonic note                  = log ratio is increasing on tau < 1/8
small-tau certified?            = True

LARGE-TAU REGIME
----------------------------------------------------------------------------------------------------
Claim: for tau >= 0.4, R(tau) <= 1.
eta_bound at split              = 0.000000415927548931693969257672233075
log margin at tau=0.4           = 8.75523752178238257715520702773
log margin positive?            = True
eta_bound <= 1?                 = True
monotonic note              

RuntimeError: theta_upper_spectral runaway

In [4]:
# ============================================================
# M5.6b — PATCHED THETA-LIMIT POINT AUDIT + CLEAN STATUS
# ============================================================
#
# Fix:
#   The prior high-precision point audit used the spectral theta sum at tiny tau.
#   That causes runaway because the spectral sum converges slowly as tau -> 0.
#
#   This patch uses:
#       spatial/Poisson theta for tau < 0.08
#       spectral theta for tau >= 0.08
#
# It also reports the adaptive middle-box certificate cleanly:
#   max rejected parent bound can be >1; only final certified boxes matter.
# ============================================================

import math
import time

try:
    import mpmath as mp
except Exception as e:
    raise RuntimeError("mpmath is required. In Colab, run: !pip install mpmath") from e

mp.mp.dps = 80

print("=" * 100)
print("M5.6b PATCHED THETA-LIMIT AUDIT")
print("=" * 100)

PI = mp.pi
SMALL_SPLIT = mp.mpf("0.05")
LARGE_SPLIT = mp.mpf("0.4")

# ------------------------------------------------------------
# Theta upper/value functions with correct Poisson switch
# ------------------------------------------------------------

def theta_upper_spatial(tau, term_tol=mp.mpf("1e-75")):
    """
    Theta(tau) = (4*pi*tau)^(-1/2) * sum_{m in Z} exp(-m^2/(4tau)).
    Efficient and stable for small tau.
    Includes a geometric tail bound.
    """
    tau = mp.mpf(tau)
    pref = 1 / mp.sqrt(4 * PI * tau)

    s = mp.mpf(1)
    m = 1

    while True:
        term = mp.e ** (-(m*m) / (4*tau))
        s += 2 * term

        if pref * term < term_tol:
            next_m = m + 1
            next_term = mp.e ** (-(next_m*next_m) / (4*tau))

            # consecutive ratio from next_m onward:
            # exp(-[(m+1)^2 - m^2]/(4tau)) = exp(-(2m+1)/(4tau))
            ratio = mp.e ** (-(2*next_m + 1) / (4*tau))
            tail = 2 * next_term / (1 - ratio)
            return pref * (s + tail)

        m += 1
        if m > 100000:
            raise RuntimeError("theta_upper_spatial runaway")

def theta_upper_spectral(tau, term_tol=mp.mpf("1e-75")):
    """
    Theta(tau) = 1 + 2 sum_{n>=1} exp(-4*pi^2*n^2*tau).
    Efficient and stable for moderate/large tau.
    Includes a geometric tail bound.
    """
    tau = mp.mpf(tau)
    lam = 4 * PI**2 * tau

    s = mp.mpf(1)
    n = 1

    while True:
        term = mp.e ** (-lam * n * n)
        s += 2 * term

        if term < term_tol:
            next_n = n + 1
            next_term = mp.e ** (-lam * next_n * next_n)
            ratio = mp.e ** (-lam * (2 * next_n + 1))
            tail = 2 * next_term / (1 - ratio)
            return s + tail

        n += 1
        if n > 100000:
            raise RuntimeError("theta_upper_spectral runaway")

def theta_upper(tau):
    tau = mp.mpf(tau)
    if tau < mp.mpf("0.08"):
        return theta_upper_spatial(tau)
    return theta_upper_spectral(tau)

def R_upper_point(tau):
    tau = mp.mpf(tau)
    th = theta_upper(tau)
    return 16 * PI**2 * tau**2 * (th**4 - 1)

# ------------------------------------------------------------
# Small/large sufficient checks
# ------------------------------------------------------------

def small_log_margin(tau):
    tau = mp.mpf(tau)
    return mp.log(16 * PI**2 * tau**2) - (mp.log(45) - 1/(4*tau))

def small_eps_bound(tau):
    tau = mp.mpf(tau)
    return 3 * mp.e ** (-1/(4*tau))

def large_log_margin(tau):
    tau = mp.mpf(tau)
    return -2 * mp.log(4 * PI * tau) - (mp.log(45) - 4 * PI**2 * tau)

def large_eta_bound(tau):
    tau = mp.mpf(tau)
    return 3 * mp.e ** (-4 * PI**2 * tau)

small_margin = small_log_margin(SMALL_SPLIT)
small_eps = small_eps_bound(SMALL_SPLIT)
large_margin = large_log_margin(LARGE_SPLIT)
large_eta = large_eta_bound(LARGE_SPLIT)

small_pass = (small_margin > 0 and small_eps <= 1)
large_pass = (large_margin > 0 and large_eta <= 1)

print("SMALL/LARGE CHECKS")
print("-" * 100)
print(f"small margin at 0.05       = {mp.nstr(small_margin, 30)}")
print(f"small eps bound at 0.05    = {mp.nstr(small_eps, 30)}")
print(f"small pass                 = {small_pass}")
print(f"large margin at 0.4        = {mp.nstr(large_margin, 30)}")
print(f"large eta bound at 0.4     = {mp.nstr(large_eta, 30)}")
print(f"large pass                 = {large_pass}")
print()

# ------------------------------------------------------------
# Middle interval adaptive certification, clean reporting
# ------------------------------------------------------------

def R_box_upper(a, b):
    """
    On [a,b]:
      theta(tau) <= theta_upper(a), because theta is decreasing.
      tau^2 <= b^2.
    Hence:
      R(tau) <= 16*pi^2*b^2*(theta_upper(a)^4 - 1).
    """
    a = mp.mpf(a)
    b = mp.mpf(b)
    th = theta_upper(a)
    return 16 * PI**2 * b**2 * (th**4 - 1)

print("MIDDLE BOX CERTIFICATE")
print("-" * 100)

start = time.time()

stack = [(SMALL_SPLIT, LARGE_SPLIT)]
certified = []
rejected_parent_count = 0
max_certified_upper = mp.mpf("-inf")
max_certified_box = None
max_rejected_parent_upper = mp.mpf("-inf")
max_rejected_parent_box = None

MAX_BOXES = 2_000_000
MIN_WIDTH = mp.mpf("1e-18")
failed = []

while stack:
    a, b = stack.pop()
    upper = R_box_upper(a, b)

    if upper <= 1:
        certified.append((a, b, upper))

        if upper > max_certified_upper:
            max_certified_upper = upper
            max_certified_box = (a, b, upper)
        continue

    rejected_parent_count += 1

    if upper > max_rejected_parent_upper:
        max_rejected_parent_upper = upper
        max_rejected_parent_box = (a, b, upper)

    width = b - a

    if width <= MIN_WIDTH or len(certified) + len(stack) > MAX_BOXES:
        failed.append((a, b, upper))
        break

    mid = (a + b) / 2
    stack.append((mid, b))
    stack.append((a, mid))

elapsed = time.time() - start
middle_pass = (len(failed) == 0)

print(f"middle pass                 = {middle_pass}")
print(f"certified boxes             = {len(certified)}")
print(f"rejected parent boxes split = {rejected_parent_count}")
print(f"failed boxes                = {len(failed)}")
print(f"elapsed seconds             = {elapsed:.4f}")
print(f"max certified upper R       = {mp.nstr(max_certified_upper, 30)}")
print(f"max certified box           = [{mp.nstr(max_certified_box[0], 18)}, {mp.nstr(max_certified_box[1], 18)}]")
print(f"largest rejected parent R   = {mp.nstr(max_rejected_parent_upper, 30)}")
print(f"largest rejected parent box = [{mp.nstr(max_rejected_parent_box[0], 18)}, {mp.nstr(max_rejected_parent_box[1], 18)}]")
print()

if failed:
    print("FAILED BOX")
    print("-" * 100)
    a, b, u = failed[0]
    print(f"a     = {mp.nstr(a, 30)}")
    print(f"b     = {mp.nstr(b, 30)}")
    print(f"upper = {mp.nstr(u, 30)}")
    print()

# ------------------------------------------------------------
# High-precision point audit with correct switch
# ------------------------------------------------------------

print("PATCHED HIGH-PRECISION POINT AUDIT")
print("-" * 100)

audit_points = [
    mp.mpf("1e-12"),
    mp.mpf("1e-9"),
    mp.mpf("1e-6"),
    mp.mpf("1e-4"),
    mp.mpf("0.001"),
    mp.mpf("0.005"),
    mp.mpf("0.01"),
    mp.mpf("0.02"),
    mp.mpf("0.03"),
    mp.mpf("0.04"),
    mp.mpf("0.05"),
    mp.mpf("0.075"),
    mp.mpf("0.10"),
    mp.mpf("0.12"),
    mp.mpf("0.20"),
    mp.mpf("0.40"),
    mp.mpf("1.0"),
    mp.mpf("10.0"),
    mp.mpf("100.0"),
]

max_point_R = mp.mpf("-inf")
max_point_tau = None

for tau in audit_points:
    rv = R_upper_point(tau)

    if rv > max_point_R:
        max_point_R = rv
        max_point_tau = tau

    print(
        f"tau={mp.nstr(tau, 12):>14s}  "
        f"R_upper={mp.nstr(rv, 35):>42s}  "
        f"1-R={mp.nstr(1-rv, 25)}"
    )

print()
print(f"max point R_upper          = {mp.nstr(max_point_R, 35)}")
print(f"at tau                     = {mp.nstr(max_point_tau, 20)}")
print()

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

overall = small_pass and middle_pass and large_pass

print("=" * 100)
print("M5.6b RESULT")
print("=" * 100)
print(f"small regime certified?       = {small_pass}")
print(f"middle regime certified?      = {middle_pass}")
print(f"large regime certified?       = {large_pass}")
print(f"theta-limit boxed?            = {overall}")
print()

if overall:
    print("Certificate-level theta-limit statement:")
    print()
    print("    16*pi^2*tau^2*(Theta(tau)^4 - 1) <= 1")
    print("    for all tau > 0.")
    print()
    print("This closes the continuum theta obstruction.")
    print("It still does not prove the finite-L cover-domination lemma.")
else:
    print("The theta-limit certificate did not close. Inspect failed boxes/margins.")

print()
print("REMAINING M5 PIN")
print("-" * 100)
print("Prove finite-cycle cover domination:")
print()
print("    Phi_L(t)^4 - L^-4 <= q(t)^4")
print()
print("for every integer L >= 4 and every t > 0.")
print()
print("With M5.4 C_inf_star < 0.013, this would imply:")
print("    T_C < 1/8,  N*_C >= 8,  T_full < 9/64,  N* >= 7.")
print("=" * 100)

M5.6b PATCHED THETA-LIMIT AUDIT
SMALL/LARGE CHECKS
----------------------------------------------------------------------------------------------------
small margin at 0.05       = 0.263921457060279841694086229183
small eps bound at 0.05    = 0.0202138409972564012899081452694
small pass                 = True
large margin at 0.4        = 8.75523752178238257715520702773
large eta bound at 0.4     = 0.000000415927548931693969257672233075
large pass                 = True

MIDDLE BOX CERTIFICATE
----------------------------------------------------------------------------------------------------
middle pass                 = True
certified boxes             = 6
rejected parent boxes split = 5
failed boxes                = 0
elapsed seconds             = 0.0064
max certified upper R       = 0.980657081547262303977333097357
max certified box           = [0.05, 0.0609375]
largest rejected parent R   = 42.2540055859838148280463523132
largest rejected parent box = [0.05, 0.4]

PATCHED HIGH-PREC

In [5]:
# ============================================================
# M5.7 — FINITE-L COVER-DOMINATION MAXIMIZER AUDIT
# ============================================================
#
# Target finite-L lemma:
#
#   Phi_L(t)^4 - L^-4 <= q(t)^4
#
# Equivalent:
#
#   D_L(t) := Phi_L(t)^4 - q(t)^4 - L^-4 <= 0.
#
# This block maximizes D_L(t) over t>0 for each L.
#
# Scaling:
#   t = tau L^2.
#
# For fixed L, D_L(t) -> 0^- as t -> infinity and D_L(t) -> 0^- as t -> 0.
# The potential danger is a finite diffusive window tau ~ O(1).
#
# This is a diagnostic, not a formal proof.
# It tells us where a finite-L interval proof has to focus.
# ============================================================

import math
import time
import numpy as np

try:
    from scipy.special import ive
    from scipy.optimize import minimize_scalar
except Exception as e:
    raise RuntimeError("SciPy is required. In Colab, run: !pip install scipy") from e

print("=" * 100)
print("M5.7 FINITE-L COVER-DOMINATION MAXIMIZER AUDIT")
print("=" * 100)

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

L_MAX = 2048

# Optimize over y = log tau.
# tau from exp(-30) to exp(20) is absurdly wide:
#   tiny t through far post-mixing.
Y_MIN = -30.0
Y_MAX = 20.0

BESSEL_REL_CUTOFF = 1e-16

print(f"L_MAX             = {L_MAX}")
print(f"y range           = [{Y_MIN}, {Y_MAX}], tau=exp(y)")
print()

# ------------------------------------------------------------
# Heat kernels
# ------------------------------------------------------------

def q0(t):
    return float(ive(0, 2.0*t))

def Phi_L(L, t):
    out = float(ive(0, 2.0*t))
    j = 1

    while True:
        term = 2.0 * float(ive(j*L, 2.0*t))
        out_new = out + term

        if term <= BESSEL_REL_CUTOFF * max(abs(out_new), 1e-300):
            return out_new

        out = out_new
        j += 1

        if j > 300000:
            raise RuntimeError(f"Phi_L runaway: L={L}, t={t}, out={out}")

def D_of_y(L, y):
    tau = math.exp(y)
    t = tau * L * L

    ph = Phi_L(L, t)
    q = q0(t)

    return ph**4 - q**4 - L**-4

def scaled_E_of_y(L, y):
    """
    Scale out L^-4:
        E_L(tau) = L^4*(Phi^4 - q^4) - 1.
    Lemma A is E_L <= 0.
    """
    return (L**4) * (D_of_y(L, y))

# ------------------------------------------------------------
# Robust maximize over y by interval subdivision
# ------------------------------------------------------------

def maximize_for_L(L):
    """
    Maximize E_L(y) over y in [Y_MIN,Y_MAX].
    Uses coarse grid to find candidate basins, then bounded scalar optimization.
    """
    # Enough to catch multiple basins if any exist.
    grid = np.linspace(Y_MIN, Y_MAX, 220)
    vals = np.array([scaled_E_of_y(L, float(y)) for y in grid])

    # Candidate intervals around local maxima and endpoints.
    candidate_intervals = []

    # endpoints
    candidate_intervals.append((grid[0], grid[1]))
    candidate_intervals.append((grid[-2], grid[-1]))

    for i in range(1, len(grid)-1):
        if vals[i] >= vals[i-1] and vals[i] >= vals[i+1]:
            a = grid[max(0, i-2)]
            b = grid[min(len(grid)-1, i+2)]
            candidate_intervals.append((float(a), float(b)))

    best = {
        "L": L,
        "Emax": -float("inf"),
        "Dmax": -float("inf"),
        "y": None,
        "tau": None,
        "t": None,
        "Phi": None,
        "q0": None,
        "method": None,
    }

    for a, b in candidate_intervals:
        def negE(y):
            return -scaled_E_of_y(L, y)

        res = minimize_scalar(
            negE,
            bounds=(a, b),
            method="bounded",
            options={"xatol": 1e-11}
        )

        ystar = float(res.x)
        Emax = -float(res.fun)
        tau = math.exp(ystar)
        t = tau * L * L
        ph = Phi_L(L, t)
        q = q0(t)
        D = ph**4 - q**4 - L**-4

        if Emax > best["Emax"]:
            best.update({
                "Emax": Emax,
                "Dmax": D,
                "y": ystar,
                "tau": tau,
                "t": t,
                "Phi": ph,
                "q0": q,
                "method": "opt",
            })

    return best

# ------------------------------------------------------------
# Main scan
# ------------------------------------------------------------

start = time.time()

rows = []
worst = None

# Scan all L up to L_MAX.
# This is faster than it looks because Phi_L image sums truncate rapidly.
for L in range(4, L_MAX + 1):
    r = maximize_for_L(L)
    rows.append(r)

    if worst is None or r["Emax"] > worst["Emax"]:
        worst = r

    if L in [4,5,6,7,8,10,12,16,24,32,47,64,96,128,192,256,384,512,768,1024,1536,2048]:
        print(
            f"L={L:5d}  "
            f"Emax={r['Emax']:.15e}  "
            f"Dmax={r['Dmax']:.15e}  "
            f"tau*={r['tau']:.9e}  "
            f"t*={r['t']:.9e}  "
            f"Phi={r['Phi']:.9e}  "
            f"q0={r['q0']:.9e}"
        )

elapsed = time.time() - start

print()
print("=" * 100)
print("FINITE-L COVER MAX SUMMARY")
print("=" * 100)
print(f"elapsed seconds      = {elapsed:.2f}")
print(f"scanned L            = 4 ... {L_MAX}")
print()
print("Worst row:")
print({
    "L": worst["L"],
    "Emax": worst["Emax"],
    "Dmax": worst["Dmax"],
    "tau_star": worst["tau"],
    "t_star": worst["t"],
    "Phi": worst["Phi"],
    "q0": worst["q0"],
})
print()
print(f"finite scan pass?    = {all(r['Emax'] <= 1e-10 for r in rows)}")
print()

# ------------------------------------------------------------
# Shape summary: does Emax approach theta-limit 0 from below?
# ------------------------------------------------------------

print("=" * 100)
print("ASYMPTOTIC SHAPE SUMMARY")
print("=" * 100)

# Print the largest Emax rows.
top = sorted(rows, key=lambda r: r["Emax"], reverse=True)[:20]

print(f"{'rank':>4} {'L':>6} {'Emax':>20} {'tau*':>15} {'t*/L^2':>15}")
for i, r in enumerate(top, 1):
    print(
        f"{i:4d} "
        f"{r['L']:6d} "
        f"{r['Emax']:20.12e} "
        f"{r['tau']:15.8e} "
        f"{(r['t']/(r['L']*r['L'])):15.8e}"
    )

print()

# ------------------------------------------------------------
# Optional tail extrapolation
# ------------------------------------------------------------

print("=" * 100)
print("TAIL EXTRAPOLATION DIAGNOSTIC")
print("=" * 100)

# Fit -Emax ~ c/L^p over large L if Emax<0.
tail_rows = [r for r in rows if r["L"] >= max(64, L_MAX//4) and r["Emax"] < 0]

if len(tail_rows) >= 10:
    X = np.log(np.array([r["L"] for r in tail_rows], dtype=float))
    Y = np.log(np.array([-r["Emax"] for r in tail_rows], dtype=float))

    p, logc = np.polyfit(X, Y, 1)
    cfit = math.exp(logc)

    print(f"Fit log(-Emax) = log c + p log L over L >= {tail_rows[0]['L']}")
    print(f"p              = {p:.6f}")
    print(f"c              = {cfit:.6e}")
    print()
    print("Interpretation:")
    print("  If p<0 and stable, Emax approaches 0 from below as L grows.")
    print("  That matches the theta-limit inequality, where the supremum is 0 at tau->0.")
else:
    print("Not enough negative tail rows for fit.")

print()
print("M5.7 STATUS")
print("-" * 100)
print("If every Emax is <= 0 up to large L and the worst cases approach 0 from below,")
print("then the finite-L pin is likely true but still needs an analytic proof.")
print()
print("Next proof target after this diagnostic:")
print("  prove the image-tail form")
print("      Phi_L(t)^4 - q0(t)^4 <= L^-4")
print("  using Bessel/Skellam tail bounds plus the boxed theta-limit inequality.")
print("=" * 100)

M5.7 FINITE-L COVER-DOMINATION MAXIMIZER AUDIT
L_MAX             = 2048
y range           = [-30.0, 20.0], tau=exp(y)



RuntimeError: Phi_L runaway: L=4, t=629963977.488632, out=nan

In [6]:
# ============================================================
# M5.7b — FINITE-L COVER-DOMINATION MAXIMIZER AUDIT, STABLE HYBRID
# ============================================================
#
# Fix over M5.7:
#   Use the exact spectral cycle heat kernel when t/L^2 is not small.
#   Use the Bessel/image form only in the small-t regime.
#
# Target:
#   D_L(t) := Phi_L(t)^4 - q(t)^4 - L^-4 <= 0.
#
# Stable scaled form:
#   E_L(t) := L^4 D_L(t)
#           = (L Phi_L(t))^4 - 1 - (L q(t))^4.
#
# Lemma A is E_L(t) <= 0.
# ============================================================

import math
import time
import numpy as np

try:
    from scipy.special import ive
    from scipy.optimize import minimize_scalar
except Exception as e:
    raise RuntimeError("SciPy is required. In Colab, run: !pip install scipy") from e

print("=" * 100)
print("M5.7b FINITE-L COVER-DOMINATION MAXIMIZER AUDIT, STABLE HYBRID")
print("=" * 100)

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

L_MAX = 2048

# Work in y = log(tau), tau = t/L^2.
# Very large y is unnecessary; post-mixing E approaches 0^-.
Y_MIN = -30.0
Y_MAX = 12.0

BESSEL_REL_CUTOFF = 1e-16

# Switch to spectral representation once tau=t/L^2 is moderate.
# This avoids Bessel overflow/nan for huge t.
TAU_SWITCH = 0.04

print(f"L_MAX             = {L_MAX}")
print(f"y range           = [{Y_MIN}, {Y_MAX}], tau=exp(y)")
print(f"TAU_SWITCH        = {TAU_SWITCH}")
print()

# ------------------------------------------------------------
# Infinite-line q(t)=e^-2t I_0(2t), stable
# ------------------------------------------------------------

def q0_asymptotic(t):
    """
    e^-2t I_0(2t) asymptotic for large t:
      I0(z)e^-z ~ 1/sqrt(2*pi*z) * [1 + 1/(8z) + 9/(128z^2) + 225/(3072z^3)]
    with z=2t.
    """
    if t <= 0:
        return 1.0

    inv = 1.0 / t
    corr = (
        1.0
        + inv / 16.0
        + 9.0 * inv * inv / 512.0
        + 225.0 * inv**3 / 24576.0
    )
    return corr / math.sqrt(4.0 * math.pi * t)

def q0(t):
    if t < 1.0e6:
        v = float(ive(0, 2.0*t))
        if math.isfinite(v):
            return v
    return q0_asymptotic(t)

# ------------------------------------------------------------
# Cycle heat-kernel Phi_L(t), hybrid exact forms
# ------------------------------------------------------------

def Phi_L_image(L, t):
    """
    Exact image/Bessel representation:
      Phi_L(t)=sum_{j in Z} e^-2t I_{jL}(2t).
    Best when tau=t/L^2 is small.
    """
    out = float(ive(0, 2.0*t))
    if not math.isfinite(out):
        raise RuntimeError(f"image base nan: L={L}, t={t}")

    j = 1
    while True:
        term = 2.0 * float(ive(j*L, 2.0*t))

        if not math.isfinite(term):
            raise RuntimeError(f"image term nan: L={L}, t={t}, j={j}")

        out_new = out + term

        if term <= BESSEL_REL_CUTOFF * max(abs(out_new), 1e-300):
            return out_new

        out = out_new
        j += 1

        if j > 300000:
            raise RuntimeError(f"Phi_L_image runaway: L={L}, t={t}, out={out}")

def Phi_L_spectral(L, t):
    """
    Exact finite-cycle spectral representation:
      Phi_L(t) = (1/L) sum_{k=0}^{L-1} exp[-4t sin^2(pi k/L)].

    Reduced by k <-> L-k symmetry. Stable when tau=t/L^2 is moderate/large.
    """
    total = 1.0  # k=0

    half = L // 2

    # paired k=1,... except Nyquist if L even
    last_pair = half if (L % 2 == 1) else half - 1

    for k in range(1, last_pair + 1):
        s = math.sin(math.pi * k / L)
        exponent = -4.0 * t * s * s

        # exp(-745) is effectively underflow.
        if exponent < -745.0:
            # terms only get smaller until k=L/2
            break

        total += 2.0 * math.exp(exponent)

    if L % 2 == 0:
        exponent = -4.0 * t
        if exponent > -745.0:
            total += math.exp(exponent)

    return total / float(L)

def Phi_L(L, t):
    tau = t / (L * L)

    if tau < TAU_SWITCH:
        return Phi_L_image(L, t)
    else:
        return Phi_L_spectral(L, t)

# ------------------------------------------------------------
# Stable scaled E_L
# ------------------------------------------------------------

def E_of_y(L, y):
    tau = math.exp(y)
    t = tau * L * L

    ph = Phi_L(L, t)
    q = q0(t)

    r = L * ph
    s = L * q

    # E = r^4 - 1 - s^4.
    # Stable near r=1.
    if r > 0:
        r4_minus_1 = math.expm1(4.0 * math.log(r))
    else:
        r4_minus_1 = -1.0

    return r4_minus_1 - s**4

# ------------------------------------------------------------
# Maximize over y
# ------------------------------------------------------------

def maximize_for_L(L):
    # coarse scan
    grid = np.linspace(Y_MIN, Y_MAX, 260)
    vals = np.array([E_of_y(L, float(y)) for y in grid])

    candidate_intervals = []

    # endpoints
    candidate_intervals.append((grid[0], grid[1]))
    candidate_intervals.append((grid[-2], grid[-1]))

    # local maxima
    for i in range(1, len(grid)-1):
        if vals[i] >= vals[i-1] and vals[i] >= vals[i+1]:
            a = grid[max(0, i-2)]
            b = grid[min(len(grid)-1, i+2)]
            candidate_intervals.append((float(a), float(b)))

    best = {
        "L": L,
        "Emax": -float("inf"),
        "y": None,
        "tau": None,
        "t": None,
        "Phi": None,
        "q0": None,
    }

    for a, b in candidate_intervals:
        def negE(y):
            return -E_of_y(L, y)

        res = minimize_scalar(
            negE,
            bounds=(a, b),
            method="bounded",
            options={"xatol": 2e-11}
        )

        ystar = float(res.x)
        emax = -float(res.fun)
        tau = math.exp(ystar)
        t = tau * L * L
        ph = Phi_L(L, t)
        q = q0(t)

        if emax > best["Emax"]:
            best.update({
                "Emax": emax,
                "y": ystar,
                "tau": tau,
                "t": t,
                "Phi": ph,
                "q0": q,
            })

    return best

# ------------------------------------------------------------
# Main scan
# ------------------------------------------------------------

start = time.time()

rows = []
worst = None

for L in range(4, L_MAX + 1):
    r = maximize_for_L(L)
    rows.append(r)

    if worst is None or r["Emax"] > worst["Emax"]:
        worst = r

    if L in [4,5,6,7,8,10,12,16,24,32,47,64,96,128,192,256,384,512,768,1024,1536,2048]:
        print(
            f"L={L:5d}  "
            f"Emax={r['Emax']:.15e}  "
            f"tau*={r['tau']:.9e}  "
            f"t*={r['t']:.9e}  "
            f"Phi={r['Phi']:.9e}  "
            f"q0={r['q0']:.9e}"
        )

elapsed = time.time() - start

print()
print("=" * 100)
print("FINITE-L COVER MAX SUMMARY")
print("=" * 100)
print(f"elapsed seconds      = {elapsed:.2f}")
print(f"scanned L            = 4 ... {L_MAX}")
print()
print("Worst row:")
print({
    "L": worst["L"],
    "Emax": worst["Emax"],
    "tau_star": worst["tau"],
    "t_star": worst["t"],
    "Phi": worst["Phi"],
    "q0": worst["q0"],
})
print()
print(f"finite scan pass?    = {all(r['Emax'] <= 1e-10 for r in rows)}")
print()

# ------------------------------------------------------------
# Top rows
# ------------------------------------------------------------

print("=" * 100)
print("TOP 20 Emax ROWS")
print("=" * 100)

top = sorted(rows, key=lambda r: r["Emax"], reverse=True)[:20]

print(f"{'rank':>4} {'L':>6} {'Emax':>22} {'tau*':>16} {'t*':>16}")
for i, r in enumerate(top, 1):
    print(
        f"{i:4d} "
        f"{r['L']:6d} "
        f"{r['Emax']:22.14e} "
        f"{r['tau']:16.9e} "
        f"{r['t']:16.9e}"
    )

print()

# ------------------------------------------------------------
# Tail extrapolation
# ------------------------------------------------------------

print("=" * 100)
print("TAIL EXTRAPOLATION DIAGNOSTIC")
print("=" * 100)

tail_rows = [r for r in rows if r["L"] >= max(64, L_MAX//4) and r["Emax"] < 0]

if len(tail_rows) >= 10:
    X = np.log(np.array([r["L"] for r in tail_rows], dtype=float))
    Y = np.log(np.array([-r["Emax"] for r in tail_rows], dtype=float))

    slope, intercept = np.polyfit(X, Y, 1)
    cfit = math.exp(intercept)

    print(f"Fit log(-Emax) = log c + p log L over L >= {tail_rows[0]['L']}")
    print(f"p              = {slope:.6f}")
    print(f"c              = {cfit:.6e}")
else:
    print("Not enough negative tail rows for fit.")

print()

# ------------------------------------------------------------
# Regime sanity: check if maximum is at far right boundary
# ------------------------------------------------------------

right_hits = [r for r in rows if abs(r["y"] - Y_MAX) < 1e-5]
left_hits = [r for r in rows if abs(r["y"] - Y_MIN) < 1e-5]

print("=" * 100)
print("BOUNDARY CHECK")
print("=" * 100)
print(f"left-boundary hits   = {len(left_hits)}")
print(f"right-boundary hits  = {len(right_hits)}")
print("If right-boundary hits occur, increase Y_MAX or use asymptotic post-mixing proof.")
print()

print("M5.7b STATUS")
print("-" * 100)
print("If every Emax is <= 0, the finite-L cover-domination lemma survives this audit.")
print("The remaining proof target is still analytic:")
print()
print("    Phi_L(t)^4 - L^-4 <= q(t)^4")
print()
print("for all integer L >= 4 and t > 0.")
print("=" * 100)

M5.7b FINITE-L COVER-DOMINATION MAXIMIZER AUDIT, STABLE HYBRID
L_MAX             = 2048
y range           = [-30.0, 12.0], tau=exp(y)
TAU_SWITCH        = 0.04

L=    4  Emax=-2.390633021517494e-13  tau*=1.627547603e+05  t*=2.604076165e+06  Phi=2.500000000e-01  q0=1.748107907e-04
L=    5  Emax=-2.390632938894333e-13  tau*=1.627547603e+05  t*=4.068869007e+06  Phi=2.000000000e-01  q0=1.398486313e-04
L=    6  Emax=-2.390632894012622e-13  tau*=1.627547603e+05  t*=5.859171371e+06  Phi=1.666666667e-01  q0=1.165405256e-04
L=    7  Emax=-2.390632866950365e-13  tau*=1.627547603e+05  t*=7.974983255e+06  Phi=1.428571429e-01  q0=9.989187878e-05
L=    8  Emax=-2.390632849385920e-13  tau*=1.627547603e+05  t*=1.041630466e+07  Phi=1.250000000e-01  q0=8.740539377e-05
L=   10  Emax=-2.390632828730136e-13  tau*=1.627547603e+05  t*=1.627547603e+07  Phi=1.000000000e-01  q0=6.992431487e-05
L=   12  Emax=-2.390632817509707e-13  tau*=1.627547603e+05  t*=2.343668548e+07  Phi=8.333333333e-02  q0=5.827026232e-05


In [7]:
# ============================================================
# M5.8 — FINITE-L COVER PROOF SPLIT AUDIT
# ============================================================
#
# Target finite-L cover lemma:
#
#   Phi_L(t)^4 - L^-4 <= q(t)^4.
#
# Scaled:
#
#   E_L(t) := L^4(Phi_L(t)^4 - q(t)^4) - 1 <= 0.
#
# M5.7b found the global max rides t -> infinity, where E_L -> 0^-.
# So split:
#
#   A. Post-mixing tau=t/L^2 >= tau0:
#      prove by spectral-gap decay:
#
#        e := L Phi_L - 1
#           <= 2 sum_{k>=1} exp(-16 tau k^2)
#
#      and lower-bound L q(t). If
#
#        (1+e)^4 - 1 <= (Lq)^4,
#
#      then E_L <= 0.
#
#   B. Compact tau in (0,tau0]:
#      maximize E_L directly, using cancellation-safe image-tail formula
#      when tau is small.
#
# This is still an audit, but it identifies a realistic proof route.
# ============================================================

import math
import time
import numpy as np

try:
    from scipy.special import ive
    from scipy.optimize import minimize_scalar
except Exception as e:
    raise RuntimeError("SciPy is required. In Colab, run: !pip install scipy") from e

print("=" * 100)
print("M5.8 FINITE-L COVER PROOF SPLIT AUDIT")
print("=" * 100)

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

L_MAX = 2048
TAU0 = 0.4
TAU_SWITCH = 0.04

Y_MIN = -30.0
Y_MAX_COMPACT = math.log(TAU0)

BESSEL_REL_CUTOFF = 1e-16

print(f"L_MAX             = {L_MAX}")
print(f"TAU0 post-mixing  = {TAU0}")
print(f"TAU_SWITCH        = {TAU_SWITCH}")
print(f"compact y range   = [{Y_MIN}, {Y_MAX_COMPACT}]")
print()

# ------------------------------------------------------------
# q(t)=e^-2t I0(2t), with asymptotic fallback
# ------------------------------------------------------------

def q0_asymptotic(t):
    if t <= 0:
        return 1.0
    inv = 1.0 / t
    corr = 1.0 + inv/16.0 + 9.0*inv*inv/512.0 + 225.0*inv**3/24576.0
    return corr / math.sqrt(4.0 * math.pi * t)

def q0(t):
    if t < 1e6:
        v = float(ive(0, 2.0*t))
        if math.isfinite(v):
            return v
    return q0_asymptotic(t)

# ------------------------------------------------------------
# Image-tail form for small tau
# ------------------------------------------------------------

def image_tail_S(L, t):
    """
    S = Phi_L(t) - q0(t) = 2 sum_{j>=1} e^-2t I_{jL}(2t).
    """
    S = 0.0
    j = 1

    while True:
        term = 2.0 * float(ive(j*L, 2.0*t))

        if not math.isfinite(term):
            raise RuntimeError(f"image tail nan: L={L}, t={t}, j={j}")

        S_new = S + term

        if term <= BESSEL_REL_CUTOFF * max(abs(S_new), 1e-300):
            return S_new

        S = S_new
        j += 1

        if j > 300000:
            raise RuntimeError(f"image_tail_S runaway: L={L}, t={t}, S={S}")

def E_small_tau(L, t):
    """
    Cancellation-safe:
      Phi = q + S.
      E = L^4[(q+S)^4 - q^4] - 1.
    Let r=Lq and a=LS:
      E = 4r^3a + 6r^2a^2 + 4ra^3 + a^4 - 1.
    """
    q = q0(t)
    S = image_tail_S(L, t)

    r = L * q
    a = L * S

    return 4*r**3*a + 6*r*r*a*a + 4*r*a**3 + a**4 - 1.0

# ------------------------------------------------------------
# Spectral Phi for moderate tau
# ------------------------------------------------------------

def Phi_L_spectral(L, t):
    total = 1.0
    half = L // 2
    last_pair = half if (L % 2 == 1) else half - 1

    for k in range(1, last_pair + 1):
        s = math.sin(math.pi * k / L)
        exponent = -4.0 * t * s * s

        if exponent < -745.0:
            break

        total += 2.0 * math.exp(exponent)

    if L % 2 == 0:
        exponent = -4.0 * t
        if exponent > -745.0:
            total += math.exp(exponent)

    return total / float(L)

def E_spectral_tau(L, t):
    ph = Phi_L_spectral(L, t)
    q = q0(t)

    r = L * ph
    s = L * q

    # E = r^4 - 1 - s^4
    r4_minus_1 = math.expm1(4.0 * math.log(r)) if r > 0 else -1.0
    return r4_minus_1 - s**4

def E_of_y(L, y):
    tau = math.exp(y)
    t = tau * L * L

    if tau < TAU_SWITCH:
        return E_small_tau(L, t)
    else:
        return E_spectral_tau(L, t)

# ------------------------------------------------------------
# Post-mixing analytic sufficient bound audit
# ------------------------------------------------------------

def e_bound_tau(tau):
    """
    e = L Phi - 1 <= 2 sum_{k>=1} exp(-16 tau k^2).
    """
    s = 0.0
    k = 1
    while True:
        term = 2.0 * math.exp(-16.0 * tau * k * k)
        s += term
        if term < 1e-18 * max(1.0, abs(s)):
            return s
        k += 1
        if k > 100000:
            raise RuntimeError("e_bound_tau runaway")

def Lq_lower_asymptotic(L, tau):
    """
    Conservative lower bound proxy for L*q0(t), t=tau L^2.
    For audit: use exact q0 value.
    Formal proof can replace this with a known Bessel lower bound.
    """
    t = tau * L * L
    return L * q0(t)

def postmix_margin(L, tau):
    """
    Positive margin proves:
       (1+e)^4 - 1 <= (Lq)^4.
    """
    eb = e_bound_tau(tau)
    lq = Lq_lower_asymptotic(L, tau)
    rhs = lq**4
    lhs = (1.0 + eb)**4 - 1.0
    return rhs - lhs, eb, lq, lhs, rhs

print("POST-MIXING SUFFICIENT BOUND CHECK")
print("-" * 100)

post_Ls = [4,5,6,8,12,16,24,32,47,64,128,256,512,1024,2048]
post_taus = [0.4, 0.5, 0.75, 1.0, 2.0, 4.0]

post_min = None

print(f"{'L':>6} {'tau':>8} {'margin':>16} {'e_bound':>14} {'Lq':>14} {'pass':>6}")
for L in post_Ls:
    for tau in post_taus:
        margin, eb, lq, lhs, rhs = postmix_margin(L, tau)
        ok = margin >= 0

        if post_min is None or margin < post_min[0]:
            post_min = (margin, L, tau, eb, lq, lhs, rhs)

        if tau == 0.4 or not ok:
            print(
                f"{L:6d} {tau:8.3f} "
                f"{margin:16.9e} "
                f"{eb:14.7e} "
                f"{lq:14.7e} "
                f"{str(ok):>6s}"
            )

print()
print("Worst postmix sampled margin:")
print({
    "margin": post_min[0],
    "L": post_min[1],
    "tau": post_min[2],
    "e_bound": post_min[3],
    "Lq": post_min[4],
    "lhs": post_min[5],
    "rhs": post_min[6],
})
print()

# ------------------------------------------------------------
# Compact maximizer audit
# ------------------------------------------------------------

def maximize_compact_for_L(L):
    grid = np.linspace(Y_MIN, Y_MAX_COMPACT, 260)
    vals = np.array([E_of_y(L, float(y)) for y in grid])

    candidate_intervals = []

    # endpoints
    candidate_intervals.append((grid[0], grid[1]))
    candidate_intervals.append((grid[-2], grid[-1]))

    # local maxima
    for i in range(1, len(grid)-1):
        if vals[i] >= vals[i-1] and vals[i] >= vals[i+1]:
            a = grid[max(0, i-2)]
            b = grid[min(len(grid)-1, i+2)]
            candidate_intervals.append((float(a), float(b)))

    best = {
        "L": L,
        "Emax": -float("inf"),
        "y": None,
        "tau": None,
        "t": None,
    }

    for a, b in candidate_intervals:
        def negE(y):
            return -E_of_y(L, y)

        res = minimize_scalar(
            negE,
            bounds=(a, b),
            method="bounded",
            options={"xatol": 2e-11}
        )

        ystar = float(res.x)
        emax = -float(res.fun)
        tau = math.exp(ystar)
        t = tau * L * L

        if emax > best["Emax"]:
            best.update({
                "Emax": emax,
                "y": ystar,
                "tau": tau,
                "t": t,
            })

    return best

print("=" * 100)
print("COMPACT WINDOW MAXIMIZER: 0 < tau <= 0.4")
print("-" * 100)

start = time.time()

rows = []
worst = None

for L in range(4, L_MAX + 1):
    r = maximize_compact_for_L(L)
    rows.append(r)

    if worst is None or r["Emax"] > worst["Emax"]:
        worst = r

    if L in [4,5,6,7,8,10,12,16,24,32,47,64,96,128,192,256,384,512,768,1024,1536,2048]:
        print(
            f"L={L:5d}  "
            f"Emax={r['Emax']:.15e}  "
            f"tau*={r['tau']:.9e}  "
            f"t*={r['t']:.9e}"
        )

elapsed = time.time() - start

print()
print("=" * 100)
print("COMPACT SUMMARY")
print("=" * 100)
print(f"elapsed seconds      = {elapsed:.2f}")
print(f"scanned L            = 4 ... {L_MAX}")
print("Worst compact row:")
print({
    "L": worst["L"],
    "Emax": worst["Emax"],
    "tau_star": worst["tau"],
    "t_star": worst["t"],
})
print(f"compact scan pass?   = {all(r['Emax'] <= 1e-10 for r in rows)}")
print()

# ------------------------------------------------------------
# Top compact rows
# ------------------------------------------------------------

print("TOP 20 COMPACT Emax ROWS")
print("-" * 100)

top = sorted(rows, key=lambda r: r["Emax"], reverse=True)[:20]

print(f"{'rank':>4} {'L':>6} {'Emax':>22} {'tau*':>16} {'t*':>16}")
for i, r in enumerate(top, 1):
    print(
        f"{i:4d} "
        f"{r['L']:6d} "
        f"{r['Emax']:22.14e} "
        f"{r['tau']:16.9e} "
        f"{r['t']:16.9e}"
    )

print()

# ------------------------------------------------------------
# Compare compact tail trend
# ------------------------------------------------------------

print("=" * 100)
print("COMPACT TAIL TREND")
print("=" * 100)

tail_rows = [r for r in rows if r["L"] >= max(64, L_MAX//4) and r["Emax"] < 0]

if len(tail_rows) >= 10:
    X = np.log(np.array([r["L"] for r in tail_rows], dtype=float))
    Y = np.log(np.array([-r["Emax"] for r in tail_rows], dtype=float))

    slope, intercept = np.polyfit(X, Y, 1)
    cfit = math.exp(intercept)

    print(f"Fit log(-Emax) = log c + p log L over L >= {tail_rows[0]['L']}")
    print(f"p              = {slope:.6f}")
    print(f"c              = {cfit:.6e}")
else:
    print("Not enough negative tail rows for fit.")

print()

print("=" * 100)
print("M5.8 STATUS")
print("=" * 100)
print("Post-mixing sufficient bound sampled pass? ",
      all(postmix_margin(L, TAU0)[0] >= 0 for L in post_Ls))
print("Compact finite scan pass?              ",
      all(r["Emax"] <= 1e-10 for r in rows))
print()
print("If both hold robustly, the proof path is:")
print()
print("  1. Prove post-mixing for tau >= 0.4 using")
print("       L Phi_L - 1 <= 2 sum exp(-16 tau k^2)")
print("     and a lower bound on L q(t).")
print()
print("  2. Prove compact tau in (0,0.4] by finite-L interval boxes")
print("     plus the boxed theta-limit inequality for the large-L tail.")
print()
print("  3. Combine with M5.4 C_inf_star < 0.013.")
print()
print("Then M5 closes:")
print("     T_C < 1/8,  N*_C >= 8,  T_full < 9/64,  N* >= 7.")
print("=" * 100)

M5.8 FINITE-L COVER PROOF SPLIT AUDIT
L_MAX             = 2048
TAU0 post-mixing  = 0.4
TAU_SWITCH        = 0.04
compact y range   = [-30.0, -0.916290731874155]

POST-MIXING SUFFICIENT BOUND CHECK
----------------------------------------------------------------------------------------------------
     L      tau           margin        e_bound             Lq   pass
     4    0.400  2.786507116e-02  3.3231146e-03  4.5059603e-01   True
     5    0.400  2.724846831e-02  3.3231146e-03  4.4890156e-01   True
     6    0.400  2.692545611e-02  3.3231146e-03  4.4800618e-01   True
     8    0.400  2.661201846e-02  3.3231146e-03  4.4713219e-01   True
    12    0.400  2.639263475e-02  3.3231146e-03  4.4651739e-01   True
    16    0.400  2.631670682e-02  3.3231146e-03  4.4630402e-01   True
    24    0.400  2.626273924e-02  3.3231146e-03  4.4615217e-01   True
    32    0.400  2.624390266e-02  3.3231146e-03  4.4609913e-01   True
    47    0.400  2.623092652e-02  3.3231146e-03  4.4606259e-01   True
   